In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
from pathlib import Path
import hashlib, urllib.request, zipfile, subprocess, sys, importlib.util

WORK = Path("/content/drive/MyDrive/MiFO")
for sub in ["data/raw/fakenewsnet", "data/raw/liar", "data/processed/step3_pilot", "data/processed/step3_full"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

print("1. Downloading FakeNewsNet CSVs to Drive...")
BASE = "https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset"
for name in ["politifact_fake.csv", "politifact_real.csv", "gossipcop_fake.csv", "gossipcop_real.csv"]:
    dest = WORK / "data/raw/fakenewsnet" / name
    if not dest.exists():
        urllib.request.urlretrieve(f"{BASE}/{name}", dest)
        print(f"   Downloaded {name}")
    else:
        print(f"   Already exists: {name}")

print("\n2. Downloading LIAR dataset to Drive...")
zpath = WORK / "data/raw/liar_dataset.zip"
if not (WORK / "data/raw/liar/train.tsv").exists():
    if not zpath.exists():
        try:
            urllib.request.urlretrieve("https://www.cs.ucsb.edu/~william/data/liar_dataset.zip", zpath)
        except Exception:
            # Fallback mirror if UCSB blocks
            urllib.request.urlretrieve("https://raw.githubusercontent.com/barun-saha/liar-plus/master/data/train.tsv", WORK / "data/raw/liar/train.tsv")
            urllib.request.urlretrieve("https://raw.githubusercontent.com/barun-saha/liar-plus/master/data/test.tsv", WORK / "data/raw/liar/test.tsv")
            urllib.request.urlretrieve("https://raw.githubusercontent.com/barun-saha/liar-plus/master/data/val.tsv", WORK / "data/raw/liar/valid.tsv")
    if zpath.exists() and not (WORK / "data/raw/liar/train.tsv").exists():
        with zipfile.ZipFile(zpath) as z:
            z.extractall(WORK / "data/raw/liar")
    print("   LIAR ready!")
else:
    print("   LIAR already exists!")

print("\n3. Installing dependencies...")
for pkg in ["sklearn", "trafilatura"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\nSUCCESS! WORK =", WORK)
print("Data is now permanently saved in your Google Drive.")

1. Downloading FakeNewsNet CSVs to Drive...
   Downloaded politifact_fake.csv
   Downloaded politifact_real.csv
   Downloaded gossipcop_fake.csv
   Downloaded gossipcop_real.csv

2. Downloading LIAR dataset to Drive...
   LIAR ready!

3. Installing dependencies...

SUCCESS! WORK = /content/drive/MyDrive/MiFO
Data is now permanently saved in your Google Drive.


In [13]:
import importlib.util, subprocess, sys
from pathlib import Path

print("platform:", sys.platform, "| cwd:", Path.cwd())
print("'C:/Users/anush/MiFO' exists:", Path("C:/Users/anush/MiFO").exists())

WORK = None
for cand in [Path("C:/Users/anush/MiFO"), Path("/mnt/c/Users/anush/MiFO"),
             Path("/content/drive/MyDrive/MiFO")]:
    if (cand / "data/raw/fakenewsnet/politifact_fake.csv").exists():
        WORK = cand; break
assert WORK is not None, "Auto-locate failed — paste the lines above back to me."
print("WORK =", WORK)

for pkg in ["sklearn", "trafilatura"]:
    if importlib.util.find_spec(pkg) is None:
        print("installing", pkg)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
        if r.returncode != 0:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "--break-system-packages", pkg])
print("deps ready")

platform: linux | cwd: /content
'C:/Users/anush/MiFO' exists: False
WORK = /content/drive/MyDrive/MiFO
deps ready


In [14]:
#Cell A1
import pandas as pd, numpy as np
from urllib.parse import urlparse

RAW = WORK / "data/raw/fakenewsnet"
frames = []
for g in ["politifact", "gossipcop"]:
    for l in ["fake", "real"]:
        df = pd.read_csv(RAW / f"{g}_{l}.csv")
        df["source_group"], df["label_name"], df["label"] = g, l, (l == "fake")
        frames.append(df)
fn = pd.concat(frames, ignore_index=True)

def domain_of(u):
    u = str(u).strip()
    if not u or u.lower() == "nan": return ""
    if not u.startswith(("http://", "https://")): u = "http://" + u
    net = urlparse(u).netloc.lower()
    return net[4:] if net.startswith("www.") else net
fn["domain"] = fn["news_url"].map(domain_of)

# minority-mass closure (owed from step 2 — expect ~3,800, near the LOO errors)
d = fn[fn["domain"] != ""].groupby("domain")["label"].agg(n="size", n_fake="sum")
print(f"Minority-label articles across domains: "
      f"{int(np.minimum(d['n_fake'], d['n'] - d['n_fake']).sum())}\n")

d["bucket"] = np.where(d["n_fake"]/d["n"] >= 0.9, "clean_fake",
              np.where(d["n_fake"]/d["n"] <= 0.1, "clean_real", "MIXED"))
fn = fn.merge(d[["bucket"]], left_on="domain", right_index=True, how="left")
fn["bucket"] = fn["bucket"].fillna("no_domain")

pool = fn[(fn["domain"] != "") & (fn["domain"] != "web.archive.org")]
pool = pool.drop_duplicates(subset="news_url")

parts = [pool[pool.source_group == "politifact"]]
for (b, l), s in pool[pool.source_group == "gossipcop"].groupby(["bucket", "label_name"]):
    parts.append(s.sample(min(len(s), 125), random_state=42))
pilot = pd.concat(parts, ignore_index=True)

print(pilot.groupby(["source_group", "bucket", "label_name"]).size().to_string())
print("\nPilot URLs:", len(pilot))
out = WORK / "data/processed/step3_pilot"; out.mkdir(parents=True, exist_ok=True)
pilot.to_csv(out / "pilot_sample.csv", index=False)

Minority-label articles across domains: 3184

source_group  bucket      label_name
gossipcop     MIXED       fake          125
                          real          125
              clean_fake  fake          125
                          real           27
              clean_real  fake          125
                          real          125
politifact    MIXED       fake           57
                          real          117
              clean_fake  fake          290
                          real            1
              clean_real  fake           12
                          real          301

Pilot URLs: 1430


In [15]:
#Cell A2
import requests, time, threading, trafilatura, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

PILOT = pd.read_csv(WORK / "data/processed/step3_pilot/pilot_sample.csv")
TXT_DIR = WORK / "data/raw/crawl_pilot"; TXT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS = WORK / "data/processed/step3_pilot/results.csv"

# EDIT: put your real email here — honest crawlers identify themselves
UA_RESEARCH = {"User-Agent": "MiFOResearchBot/0.1 (academic study; contact: YOU@EMAIL.COM)"}
UA_BROWSER = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                            "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

domain_last, lock = defaultdict(float), threading.Lock()
MIN_GAP = 2.0  # seconds between hits to the same domain

def polite_wait(dom):
    while True:
        with lock:
            if time.time() >= domain_last[dom] + MIN_GAP:
                domain_last[dom] = time.time(); return
        time.sleep(0.5)

def fetch(url):
    url = str(url).strip()
    if not url.startswith(("http://", "https://")): url = "http://" + url
    try:
        r = requests.get(url, headers=UA_RESEARCH, timeout=15, allow_redirects=True)
        if r.status_code in (403, 429, 503):   # one polite retry with browser UA
            time.sleep(3)
            r = requests.get(url, headers=UA_BROWSER, timeout=15, allow_redirects=True)
        return r
    except Exception as e:
        return None

def crawl_row(row):
    polite_wait(row["domain"])
    r = fetch(row["news_url"])
    rec = {"id": row["id"], "group": row["source_group"], "label": row["label_name"],
           "bucket": row["bucket"], "url": row["news_url"], "status": None,
           "final_url": None, "text_len": 0}
    if r is not None:
        rec["status"], rec["final_url"] = r.status_code, r.url
        if r.status_code == 200 and "html" in r.headers.get("Content-Type", "").lower():
            text = trafilatura.extract(r.text, include_comments=False,
                                       include_tables=False) or ""
            rec["text_len"] = len(text)
            if text:
                (TXT_DIR / f"{row['id']}.txt").write_text(text, encoding="utf-8")
    else:
        rec["status"] = "error"
    return rec

rows, t0 = [], time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    futs = [ex.submit(crawl_row, r) for r in PILOT.to_dict("records")]
    for i, f in enumerate(as_completed(futs), 1):
        rows.append(f.result())
        if i % 100 == 0:
            pd.DataFrame(rows).to_csv(RESULTS, index=False)
            print(f"{i}/{len(PILOT)} done ({time.time()-t0:.0f}s)")
pd.DataFrame(rows).to_csv(RESULTS, index=False)
print(f"FINISHED {len(rows)} in {(time.time()-t0)/60:.1f} min -> {RESULTS}")

100/1430 done (16s)


200/1430 done (34s)


300/1430 done (50s)


400/1430 done (63s)
500/1430 done (80s)
600/1430 done (99s)
700/1430 done (117s)


800/1430 done (139s)


...

1000/1430 done (175s)


1100/1430 done (203s)


1200/1430 done (239s)


ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None


1300/1430 done (265s)
1400/1430 done (281s)
FINISHED 1430 in 4.9 min -> /content/drive/MyDrive/MiFO/data/processed/step3_pilot/results.csv


In [16]:
#Cell C2
import pandas as pd, numpy as np
COLS = ["json_id","label","statement","subjects","speaker","job","state","party",
        "barely_true","false","half_true","mostly_true","pants_fire","context"]
liar = {s: pd.read_csv(WORK / f"data/raw/liar/{s}.tsv", sep="\t",
                       names=COLS, quoting=3) for s in ["train","valid","test"]}
tr = liar["train"].copy()
tr["words"] = tr["statement"].str.split().str.len()

print("=== Statement length (words) by class ===")
print(tr.groupby("label")["words"].agg(["count","median","mean"]).round(1)
      .sort_values("median").to_string())

print(f"\n=== Speakers: {tr['speaker'].nunique()} unique; top 5 ===")
print(tr["speaker"].value_counts().head(5).to_string())

print("\n=== Credit history (mean prior counts) by current class ===")
hist = ["barely_true","false","half_true","mostly_true","pants_fire"]
h = tr[hist].apply(pd.to_numeric, errors="coerce")
h["label"] = tr["label"].values
print(h.groupby("label").mean().round(1).to_string())

subj = tr["subjects"].fillna("").str.split(",").explode().str.strip()
print("\n=== Top subjects ===");  print(subj[subj != ""].value_counts().head(10).to_string())

=== Statement length (words) by class ===
             count  median  mean
label                           
false         1998    15.0  16.8
pants-fire     842    16.0  17.1
true          1683    16.0  17.9
barely-true   1657    17.0  18.1
mostly-true   1966    17.0  18.2
half-true     2123    18.0  18.8

=== Speakers: 2916 unique; top 5 ===
speaker
barack-obama       493
donald-trump       274
hillary-clinton    239
mitt-romney        180
scott-walker       150

=== Credit history (mean prior counts) by current class ===
             barely_true  false  half_true  mostly_true  pants_fire
label                                                              
barely-true         11.7   12.4       14.8         13.9         5.4
false               11.7   15.9       15.2         14.0         7.9
half-true           11.9   12.6       19.6         18.2         4.5
mostly-true         11.8   12.3       19.8         20.6         3.8
pants-fire          11.0   18.1       11.5          9.2        1

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score

pipe = make_pipeline(TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True),
                     LogisticRegression(max_iter=2000))
pipe.fit(liar["train"]["statement"], liar["train"]["label"])
pred = pipe.predict(liar["test"]["statement"])

maj = liar["train"]["label"].value_counts(normalize=True).iloc[0]
acc = accuracy_score(liar["test"]["label"], pred)
f1m = f1_score(liar["test"]["label"], pred, average="macro")
print(f"majority baseline: {maj:.3f}  |  TF-IDF+LR accuracy: {acc:.3f}  |  macro-F1: {f1m:.3f}\n")
print(classification_report(liar["test"]["label"], pred, digits=2))

out = WORK / "data/processed/step3_liar"; out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"y": liar["test"]["label"], "pred": pred}).to_csv(out / "lr_baseline_preds.csv", index=False)
print("saved ->", out / "lr_baseline_preds.csv")

majority baseline: 0.207  |  TF-IDF+LR accuracy: 0.252  |  macro-F1: 0.219

              precision    recall  f1-score   support

 barely-true       0.23      0.14      0.18       214
       false       0.31      0.40      0.35       250
   half-true       0.22      0.29      0.25       267
 mostly-true       0.23      0.27      0.25       249
  pants-fire       0.33      0.03      0.06        92
        true       0.25      0.21      0.23       211

    accuracy                           0.25      1283
   macro avg       0.26      0.22      0.22      1283
weighted avg       0.25      0.25      0.24      1283

saved -> /content/drive/MyDrive/MiFO/data/processed/step3_liar/lr_baseline_preds.csv


In [18]:
#Cell A3
res = pd.read_csv(WORK / "data/processed/step3_pilot/results.csv")
res["live"] = res["status"] == 200
res["extracted"] = res["text_len"] >= 200

print(f"HTTP 200: {res['live'].mean():.1%}  |  extracted(>=200 chars): "
      f"{res['extracted'].mean():.1%}  |  median text len: "
      f"{res.loc[res.extracted, 'text_len'].median():.0f}")

print("\n=== By group/label ===")
print(res.groupby(["group","label"]).agg(n=("id","size"), live=("live","mean"),
      extracted=("extracted","mean")).round(3).to_string())

print("\n=== By bucket ===")
print(res.groupby("bucket").agg(n=("id","size"), live=("live","mean"),
      extracted=("extracted","mean")).round(3).to_string())

print("\n=== Status codes ===")
print(res["status"].value_counts(dropna=False).head(10).to_string())

HTTP 200: 0.0%  |  extracted(>=200 chars): 45.7%  |  median text len: 2498

=== By group/label ===
                    n  live  extracted
group      label                      
gossipcop  fake   375   0.0      0.627
           real   277   0.0      0.632
politifact fake   359   0.0      0.309
           real   419   0.0      0.317

=== By bucket ===
              n  live  extracted
bucket                          
MIXED       424   0.0      0.575
clean_fake  443   0.0      0.377
clean_real  563   0.0      0.432

=== Status codes ===
status
200      794
error    188
404      178
403      149
402       51
202       15
400       14
410       11
444        5
500        5


In [19]:
#extra A3
res["live"] = pd.to_numeric(res["status"], errors="coerce") == 200
print("Real HTTP 200 Live Rate:", res["live"].mean())

Real HTTP 200 Live Rate: 0.5552447552447553


In [20]:
#Cell A4

import requests, pandas as pd, random

res = pd.read_csv(WORK / "data/processed/step3_pilot/results.csv")
failed = res[(pd.to_numeric(res["status"], errors="coerce") != 200)
             | (res["text_len"] < 200)].copy()
sample = failed.sample(min(150, len(failed)), random_state=42)
print(f"failed/empty rows: {len(failed)}; sampling {len(sample)}")

hits = 0
for i, (_, r) in enumerate(sample.iterrows(), 1):
    try:
        api = f"http://archive.org/wayback/available?url={r['url']}"
        snap = requests.get(api, timeout=15).json().get("archived_snapshots", {}).get("closest")
        if snap and snapshot_ok(snap): hits += 1
    except Exception: pass
    if i % 50 == 0: print(f"{i}/{len(sample)} checked, {hits} snapshots")

def snapshot_ok(s): return True  # placeholder — see note below
print(f"\nWayback coverage of failed URLs: {hits}/{len(sample)} = {hits/len(sample):.1%}")

failed/empty rows: 776; sampling 150
50/150 checked, 0 snapshots
100/150 checked, 0 snapshots
150/150 checked, 0 snapshots

Wayback coverage of failed URLs: 0/150 = 0.0%


In [21]:
import requests, pandas as pd, time

res = pd.read_csv(WORK / "data/processed/step3_pilot/results.csv")
failed = res[(pd.to_numeric(res["status"], errors="coerce") != 200)
             | (res["text_len"] < 200)].copy()
sample = failed.sample(min(20, len(failed)), random_state=42)

for i, (_, r) in enumerate(sample.iterrows(), 1):
    url = str(r["url"]).strip()
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
    try:
        resp = requests.get("https://archive.org/wayback/available",
                            params={"url": url}, timeout=20)
        print(f"{i:2d}  {resp.status_code}  {resp.text[:150]}")
    except Exception as e:
        print(f"{i:2d}  EXCEPTION {type(e).__name__}: {str(e)[:80]}")
    time.sleep(2)

 1  200  {"url": "http://www.youtube.com/watch?v=Cqww8jmizug&eurl=http%3A%2F%2Fstoptheabortionagenda.com%2F&feature=player_embedded", "archived_snapshots": {"c
 2  200  {"url": "http://www.instyle.com/news/big-photo-big-rock-allison-williams-debuts-her-dazzling-engagement-ring", "archived_snapshots": {"closest": {"sta
 3  200  {"url": "http://politicalticker.blogs.cnn.com/2008/05/05/mccain-touts-gang-of-14-immigration-reform/", "archived_snapshots": {"closest": {"status": "2
 4  200  {"url": "https://www.zeel.com/massage-news/coachella-2017-all-access-go-inside-this-years-star-studded-music-festival", "archived_snapshots": {"closes
 5  200  {"url": "http://bioguide.congress.gov/scripts/biodisplay.pl?index=M000303", "archived_snapshots": {"closest": {"status": "200", "available": true, "ur
 6  200  {"url": "http://www.inquisitr.com/4578952/fergie-reportedly-crushing-hard-on-kendall-jenner-singer-wants-to-explore-fantasies-after-split-rumors/", "a
 7  200  {"url": "http://www.eefvy.co/7-

In [22]:
import pandas as pd
from urllib.parse import urlparse

RAW = WORK / "data/raw/fakenewsnet"
frames = []
for g in ["politifact", "gossipcop"]:
    for l in ["fake", "real"]:
        df = pd.read_csv(RAW / f"{g}_{l}.csv")
        df["source_group"], df["label_name"], df["label"] = g, l, (l == "fake")
        frames.append(df)
fn = pd.concat(frames, ignore_index=True)

def domain_of(u):
    u = str(u).strip()
    if not u or u.lower() == "nan": return ""
    if not u.startswith(("http://", "https://")): u = "http://" + u
    net = urlparse(u).netloc.lower()
    return net[4:] if net.startswith("www.") else net
fn["domain"] = fn["news_url"].map(domain_of)

pool = fn[(fn["domain"] != "") & (fn["domain"] != "web.archive.org")]
pool = pool.drop_duplicates(subset="news_url")

pilot_done = set(pd.read_csv(WORK / "data/processed/step3_pilot/results.csv")
                 .query("text_len >= 200")["id"])
manifest = pool[~pool["id"].isin(pilot_done)].copy()

out = WORK / "data/processed/step3_full"; out.mkdir(parents=True, exist_ok=True)
manifest.to_csv(out / "manifest.csv", index=False)
print(f"unique URLs: {len(pool)} | already have text (pilot): {len(pilot_done)}")
print(f"TO CRAWL: {len(manifest)}")
print(manifest.groupby(["source_group", "label_name"]).size().to_string())

unique URLs: 21461 | already have text (pilot): 654
TO CRAWL: 20807
source_group  label_name
gossipcop     fake           4446
              real          15827
politifact    fake            248
              real            286


In [ ]:
# F3 — fast full crawl: shuffled (parallel across domains), 1s gap, 24 workers,
# skips already-fetched articles, continuous merge. Safe to interrupt & re-run.
import requests, time, threading, trafilatura, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

UA_EMAIL = "anushaanand1543@gmail.com" 


MAN = pd.read_csv(WORK / "data/processed/step3_full/manifest.csv")
TXT = WORK / "data/raw/crawl_full"; TXT.mkdir(parents=True, exist_ok=True)
RES = WORK / "data/processed/step3_full/results.csv"

UA_R = {"User-Agent": f"MiFOResearchBot/0.1 (academic study; contact: {UA_EMAIL})"}
UA_B = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

# --- what's already done (by results.csv OR by existing .txt from a killed run)
done_ids = set(pd.read_csv(RES)["id"]) if RES.exists() else set()
have_txt = {p.stem for p in TXT.glob("*.txt")}
backfill = MAN[MAN["id"].isin([i for i in have_txt if i not in done_ids])]
if len(backfill):
    bf = pd.DataFrame({"id": backfill["id"], "group": backfill["source_group"],
        "label": backfill["label_name"], "bucket": backfill["bucket"],
        "url": backfill["news_url"], "status": 200, "final_url": None,
        "text_len": [len((TXT / f"{i}.txt").read_text(errors="ignore"))
                     for i in backfill["id"]]})
    pd.concat([pd.read_csv(RES), bf], ignore_index=True).drop_duplicates(
        subset="id", keep="last").to_csv(RES, index=False) if RES.exists() else bf.to_csv(RES, index=False)
    print(f"backfilled {len(bf)} already-fetched articles into results.csv")

done_ids = set(pd.read_csv(RES)["id"]) if RES.exists() else set()
todo = MAN[~MAN["id"].isin(done_ids)].sample(frac=1, random_state=42)  # SHUFFLE
print(f"manifest: {len(MAN)} | done: {len(done_ids)} | todo: {len(todo)}")

domain_last, lock = defaultdict(float), threading.Lock()
MIN_GAP = 1.0

def polite_wait(dom):
    while True:
        with lock:
            if time.time() >= domain_last[dom] + MIN_GAP:
                domain_last[dom] = time.time(); return
        time.sleep(0.3)

def fetch(url):
    url = str(url).strip()
    if not url.startswith(("http://", "https://")): url = "http://" + url
    try:
        r = requests.get(url, headers=UA_R, timeout=15, allow_redirects=True)
        if r.status_code in (403, 429, 503):
            time.sleep(3)
            r = requests.get(url, headers=UA_B, timeout=15, allow_redirects=True)
        return r
    except Exception:
        return None

def crawl_row(row):
    polite_wait(row["domain"])
    r = fetch(row["news_url"])
    rec = {"id": row["id"], "group": row["source_group"], "label": row["label_name"],
           "bucket": row["bucket"], "url": row["news_url"], "status": None,
           "final_url": None, "text_len": 0}
    if r is not None:
        rec["status"], rec["final_url"] = r.status_code, r.url
        if r.status_code == 200 and "html" in r.headers.get("Content-Type", "").lower():
            text = trafilatura.extract(r.text, include_comments=False,
                                       include_tables=False) or ""
            rec["text_len"] = len(text)
            if text:
                (TXT / f"{row['id']}.txt").write_text(text, encoding="utf-8")
    else:
        rec["status"] = "error"
    return rec

rows, t0 = [], time.time()
with ThreadPoolExecutor(max_workers=24) as ex:
    futs = [ex.submit(crawl_row, r) for r in todo.to_dict("records")]
    for i, f in enumerate(as_completed(futs), 1):
        rows.append(f.result())
        if i % 500 == 0:
            done = pd.concat([pd.read_csv(RES), pd.DataFrame(rows)],
                             ignore_index=True) if RES.exists() else pd.DataFrame(rows)
            done.drop_duplicates(subset="id", keep="last").to_csv(RES, index=False)
            el = (time.time() - t0) / 60
            print(f"{i}/{len(todo)}  ({el:.0f}m, ~{(len(todo)-i)*el/i:.0f}m left)")
            rows = []
if rows:
    done = pd.concat([pd.read_csv(RES), pd.DataFrame(rows)], ignore_index=True)
    done.drop_duplicates(subset="id", keep="last").to_csv(RES, index=False)

res = pd.read_csv(RES)
res["ok"] = pd.to_numeric(res["status"], errors="coerce") == 200
print(res.groupby(["group", "label"]).agg(n=("id", "size"), live=("ok", "mean"),
      with_text=("text_len", lambda s: (s >= 200).mean())).round(3).to_string())
print("\nTOTALS:", len(res), "attempted |", int((res["text_len"] >= 200).sum()), "with text")

manifest: 20807 | done: 0 | todo: 20807
